# Matched Grokking Setup: Transformer + MLP

This notebook trains two architectures on the same modular addition dataset and hyperparameters:
- single-layer single-head Transformer (Nanda/Poncini style)
- 1-hidden-layer ReLU MLP with separate left/right embeddings (Chughtai style)

The split is 30% train / 70% test by default, using a fixed seed for reproducibility.
The notebook saves checkpoints and training curves for later comparison.

## Dependencies

Run this cell if you need to install the main Python packages. If your environment already has PyTorch, NumPy, and Matplotlib, you can skip it.

In [ ]:
import math
import os
import random
import pickle
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibility
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

RUN_ROOT = os.path.join('.', 'grokking_runs')
os.makedirs(RUN_ROOT, exist_ok=True)

In [ ]:
import pickle, os, glob

def load_final_checkpoint(run_name):
    run_dir = os.path.join(RUN_ROOT, run_name)
    
    with open(os.path.join(run_dir, 'config.pkl'), 'rb') as f:
        config = pickle.load(f)
    with open(os.path.join(run_dir, 'metrics.pkl'), 'rb') as f:
        metrics = pickle.load(f)
    
    ckpt_dir = os.path.join(run_dir, 'checkpoints')
    checkpoints = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_[0-9]*.pt')))
    final_ckpt = checkpoints[-1]
    print(f'Loading: {final_ckpt}')
    
    checkpoint = torch.load(final_ckpt, map_location=DEVICE)
    print(f'Epoch: {checkpoint["epoch"]}')
    return config, checkpoint, metrics

# --- MLP ---
mlp_config, mlp_ckpt, mlp_metrics = load_final_checkpoint('mlp_run')
mlp = ChughtaiMLP(mlp_config.p, d_hidden=mlp_config.mlp_d_hidden).to(DEVICE)
mlp.load_state_dict(mlp_ckpt['model_state'])
mlp.eval()

# --- Transformer ---
transformer_config, transformer_ckpt, transformer_metrics = load_final_checkpoint('transformer_run')
transformer = GrokTransformer(transformer_config).to(DEVICE)
transformer.load_state_dict(transformer_ckpt['model_state'])
transformer.eval()

print('Both models loaded.')

In [ ]:
@dataclass(frozen=True)
class GrokConfig:
    p: int = 113
    frac_train: float = 0.30
    lr: float = 1e-3
    weight_decay: float = 1.0
    betas: tuple = (0.9, 0.98)
    d_model: int = 128
    d_mlp: int = 512
    mlp_d_hidden: int = 512
    n_ctx: int = 3
    transformer_epochs: int = 25_000
    mlp_epochs: int = 25_000
    num_checkpoints: int = 200  # roughly log-spaced model snapshots saved to disk
    seed: int = SEED
    fn_name: str = 'add'

    @property
    def d_vocab(self):
        return self.p + 1

    @property
    def fn(self):
        return lambda a, b: (a + b) % self.p

    @property
    def train_size(self):
        return int(self.frac_train * self.p * self.p)

In [ ]:
def make_dataset(config: GrokConfig):
    # Convention (Poncini): tokens are (BOS, a, b), where BOS has id `p`.
    # Loss is computed at the final position (the b position, index 2),
    # which predicts c = a + b mod p. Phase 1 will decode ⟨v^(a)| at index 1
    # and ⟨v^(b)|, ⟨v^(a)+v^(b)|, ⟨v^(a+b)| at index 2.
    pairs = [(config.p, i, j) for i in range(config.p) for j in range(config.p)]
    random.Random(config.seed).shuffle(pairs)
    labels = [config.fn(i, j) for _, i, j in pairs]
    all_data = torch.tensor(pairs, dtype=torch.long)
    all_labels = torch.tensor(labels, dtype=torch.long)
    n_train = int(config.frac_train * len(all_data))
    train_data = all_data[:n_train].to(DEVICE)
    train_labels = all_labels[:n_train].to(DEVICE)
    test_data = all_data[n_train:].to(DEVICE)
    test_labels = all_labels[n_train:].to(DEVICE)
    return train_data, train_labels, test_data, test_labels

def pretty_split(train_data, test_data):
    print(f'Train size: {len(train_data)}, Test size: {len(test_data)}')
    print(f'Train fraction: {len(train_data)/(len(train_data)+len(test_data)):.3f}')

In [ ]:
class GrokTransformer(nn.Module):
    """Single-layer, single-head transformer (Nanda/Poncini setup), no LayerNorm, no Q/K/V/O biases."""
    def __init__(self, config: GrokConfig):
        super().__init__()
        self.p = config.p
        self.vocab_size = config.d_vocab
        self.d_model = config.d_model
        self.d_mlp = config.d_mlp
        self.n_ctx = config.n_ctx
        self.embed = nn.Embedding(self.vocab_size, self.d_model)
        self.pos_embed = nn.Parameter(torch.randn(self.n_ctx, self.d_model) / math.sqrt(self.d_model))
        self.W_Q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_K = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_V = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_O = nn.Linear(self.d_model, self.d_model, bias=False)
        self.mlp_in = nn.Linear(self.d_model, self.d_mlp, bias=True)
        self.mlp_out = nn.Linear(self.d_mlp, self.d_model, bias=True)
        self.unembed = nn.Linear(self.d_model, self.vocab_size, bias=False)
        self.register_buffer('causal_mask', torch.tril(torch.ones(self.n_ctx, self.n_ctx)))

        # Match Nanda's reference init: weights ~ N(0, 1/d_model), unembed ~ N(0, 1/d_vocab),
        # MLP biases zeroed. Default nn.Linear init is Kaiming-uniform, ~sqrt(3) too small.
        nn.init.normal_(self.embed.weight, std=1.0 / math.sqrt(self.d_model))
        for layer in (self.W_Q, self.W_K, self.W_V, self.W_O, self.mlp_in, self.mlp_out):
            nn.init.normal_(layer.weight, std=1.0 / math.sqrt(self.d_model))
        nn.init.normal_(self.unembed.weight, std=1.0 / math.sqrt(self.vocab_size))
        nn.init.zeros_(self.mlp_in.bias)
        nn.init.zeros_(self.mlp_out.bias)

    def forward(self, x):
        x = self.embed(x) + self.pos_embed[None, :, :]
        q = self.W_Q(x)
        k = self.W_K(x)
        v = self.W_V(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_model // 4)
        mask = self.causal_mask[None, :, :].to(scores.dtype)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attended = torch.matmul(attn, v)
        x = x + self.W_O(attended)
        mlp = F.relu(self.mlp_in(x))
        x = x + self.mlp_out(mlp)
        return self.unembed(x)


class ChughtaiMLP(nn.Module):
    """One-hidden-layer ReLU MLP with separate left/right embeddings.

    Forward: logits = W_U @ ReLU(W_a @ a + W_b @ b)
    The summed embedding *is* the hidden layer, so embedding dim = hidden dim.
    """
    def __init__(self, p: int, d_hidden: int = 128):
        super().__init__()
        self.p = p
        self.d_hidden = d_hidden
        self.W_a = nn.Parameter(torch.randn(d_hidden, p) / math.sqrt(p))
        self.W_b = nn.Parameter(torch.randn(d_hidden, p) / math.sqrt(p))
        self.W_U = nn.Parameter(torch.randn(p, d_hidden) / math.sqrt(d_hidden))

    def forward(self, a, b):
        a_emb = F.embedding(a, self.W_a.T)  # (..., d_hidden)
        b_emb = F.embedding(b, self.W_b.T)  # (..., d_hidden)
        x = F.relu(a_emb + b_emb)
        return x @ self.W_U.T  # (..., p)

In [ ]:
def cross_entropy_high_precision(logits, labels):
    # Float64 log-softmax avoids the float32 underflow that creates dodgy
    # near-zero loss spikes during the post-grokking generalization phase.
    logprobs = F.log_softmax(logits.to(torch.float64), dim=-1)
    return -logprobs.gather(-1, labels[:, None].long()).mean()


def model_forward_logits(model, batch, is_transformer):
    if is_transformer:
        return model(batch)[:, -1, :-1]
    return model(batch[:, 1], batch[:, 2])  # columns 1 and 2 are a and b


@torch.no_grad()
def evaluate(model, data, labels, is_transformer):
    model.eval()
    logits = model_forward_logits(model, data, is_transformer)
    loss = cross_entropy_high_precision(logits, labels).item()
    accuracy = (logits.argmax(dim=-1) == labels).float().mean().item()
    return loss, accuracy


def log_spaced_epochs(num_epochs, num_points):
    # Roughly log-spaced integers in [0, num_epochs-1], plus the endpoints.
    if num_epochs <= 1:
        return {0}
    pts = np.unique(np.round(np.geomspace(1, num_epochs, num_points)).astype(int))
    pts = np.clip(pts - 1, 0, num_epochs - 1)
    return set(pts.tolist()) | {0, num_epochs - 1}


def train_loop(model, train_data, train_labels, test_data, test_labels,
               config: GrokConfig, num_epochs: int, run_name: str, is_transformer: bool):
    """Full-batch AdamW training. Records train/test loss+acc every epoch (needed for the
    grokking visualization) and saves model checkpoints at log-spaced epochs to disk.
    A tqdm progress bar shows ETA and current train/test metrics in the postfix.
    """
    optimizer = optim.AdamW(model.parameters(), lr=config.lr,
                            weight_decay=config.weight_decay, betas=config.betas)
    metrics = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    run_dir = os.path.join(RUN_ROOT, run_name)
    ckpt_dir = os.path.join(run_dir, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)
    save_epochs = log_spaced_epochs(num_epochs, config.num_checkpoints)
    postfix_every = max(1, num_epochs // 500)  # how often to refresh the bar's metric postfix

    # Save the initial (untrained) model — useful as the t=0 reference for developmental analysis.
    torch.save({'epoch': -1, 'model_state': model.state_dict()},
               os.path.join(ckpt_dir, 'epoch_init.pt'))

    pbar = tqdm(range(num_epochs), desc=run_name, dynamic_ncols=True)
    for epoch in pbar:
        model.train()
        logits = model_forward_logits(model, train_data, is_transformer)
        train_loss = cross_entropy_high_precision(logits, train_labels)
        with torch.no_grad():
            train_acc = (logits.argmax(dim=-1) == train_labels).float().mean().item()
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        test_loss, test_acc = evaluate(model, test_data, test_labels, is_transformer)
        train_loss_val = train_loss.item()
        metrics['train_loss'].append(train_loss_val)
        metrics['train_acc'].append(train_acc)
        metrics['test_loss'].append(test_loss)
        metrics['test_acc'].append(test_acc)

        if epoch in save_epochs:
            torch.save({'epoch': epoch,
                        'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict()},
                       os.path.join(ckpt_dir, f'epoch_{epoch:07d}.pt'))

        if epoch % postfix_every == 0 or epoch == num_epochs - 1:
            pbar.set_postfix(
                train_loss=f'{train_loss_val:.2e}',
                test_loss=f'{test_loss:.2e}',
                train_acc=f'{train_acc:.3f}',
                test_acc=f'{test_acc:.3f}',
            )

    pbar.close()

    with open(os.path.join(run_dir, 'metrics.pkl'), 'wb') as f:
        pickle.dump(metrics, f)
    with open(os.path.join(run_dir, 'config.pkl'), 'wb') as f:
        pickle.dump(config, f)
    print(f'Saved {len(save_epochs)} checkpoints + metrics to {run_dir}')
    return metrics

In [ ]:
config = GrokConfig()
train_data, train_labels, test_data, test_labels = make_dataset(config)
pretty_split(train_data, test_data)
print('Transformer vocabulary size:', config.d_vocab)
print('First train example:', train_data[0].tolist(), 'label', train_labels[0].item())

In [ ]:
transformer = GrokTransformer(config).to(DEVICE)
print('Transformer parameter count:', sum(p.numel() for p in transformer.parameters()))
transformer_metrics = train_loop(
    transformer, train_data, train_labels, test_data, test_labels,
    config, num_epochs=config.transformer_epochs,
    run_name='transformer_run', is_transformer=True,
)

In [ ]:
mlp = ChughtaiMLP(config.p, d_hidden=config.mlp_d_hidden).to(DEVICE)
print('MLP parameter count:', sum(p.numel() for p in mlp.parameters()))
mlp_metrics = train_loop(
    mlp, train_data, train_labels, test_data, test_labels,
    config, num_epochs=config.mlp_epochs,
    run_name='mlp_run', is_transformer=False,
)

In [ ]:
def plot_training(metrics, title):
    epochs = np.arange(len(metrics['train_loss'])) + 1
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    axs[0].plot(epochs, metrics['train_loss'], label='train')
    axs[0].plot(epochs, metrics['test_loss'], label='test')
    axs[0].set_xscale('log')
    axs[0].set_yscale('log')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Cross-entropy loss')
    axs[0].set_title(f'{title} loss')
    axs[0].legend()

    axs[1].plot(epochs, metrics['train_acc'], label='train')
    axs[1].plot(epochs, metrics['test_acc'], label='test')
    axs[1].set_xscale('log')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].set_title(f'{title} accuracy')
    axs[1].axhline(0.99, ls='--', alpha=0.4, color='gray', label='99%')
    axs[1].legend()

    plt.tight_layout()
    plt.show()


plot_training(transformer_metrics, 'Transformer')
plot_training(mlp_metrics, 'MLP')
print('Transformer final test accuracy:', transformer_metrics['test_acc'][-1])
print('MLP final test accuracy:', mlp_metrics['test_acc'][-1])